## 1. Install Dependencies

# Continue LoRA SFT Classification — Qwen3.5-0.8B, Epoch N to Epoch N+K

This notebook loads a **previously trained LoRA adapter** and continues supervised fine-tuning for **`ADDITIONAL_LORA_EPOCHS` more epochs** on the cloud-classification **single-label classification** task.

Set `INITIAL_LORA_EPOCH` (the epoch of the adapter you are loading) and `ADDITIONAL_LORA_EPOCHS` (how many epochs to add) in the Configuration cell before running.

**Classes:** the 11-class WMO-aligned taxonomy. Each descriptive label is merged into its closest WMO genus: Patterned Clouds→Altocumulus, Thick Dark Clouds→Cumulonimbus, Thick White Clouds→Cumulus, Sky (General)→Clear Sky, Veil Clouds→Veil.

**Input:** attach the Kaggle dataset that contains the previous LoRA adapter (`adapter_model.safetensors`).
**Output:** saves and zips the continued adapter under `/kaggle/working/qwen35-08-cloud-classifier-lora-epoch<FINAL>/sft_epoch<INITIAL>_to_epoch<FINAL>/lora_adapter_epoch<FINAL>`.


In [ ]:
#%pip install -U --no-cache-dir "torchao>=0.16.0"

In [ ]:
# Run this cell ONCE. When it finishes, restart the runtime
# (Runtime > Restart session) and continue from cell 2 — do NOT re-run this cell.
"""
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", *args])

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "verl", "vllm", "mistral-common", "trl",
                 "peft", "transformers", "tokenizers", "accelerate", "bitsandbytes"])

pip("install", "-U", "--no-cache-dir", "pip<25.1")

# transformers from GitHub dev (required for Qwen3.5 model_type="qwen3_5").
pip("install", "-U", "--no-cache-dir", "git+https://github.com/huggingface/transformers.git")

# peft from GitHub dev — stable PyPI peft fails with dev transformers.
pip("install", "-U", "--no-cache-dir",
    "git+https://github.com/huggingface/peft.git",
    "accelerate>=1.0.0", "datasets>=3.0.0", "safetensors>=0.4.0",
    "scikit-learn>=1.6.0", "seaborn>=0.13.0", "matplotlib>=3.8.0,<3.11.0",
    "pillow>=10.4.0,<12.0.0", "torchvision", "qwen-vl-utils>=0.0.10")

pip("install", "--force-reinstall", "--no-cache-dir",
    "numpy==2.1.3", "scipy==1.14.1", "scikit-learn==1.7.2", "pillow>=10.4.0,<12.0.0")

subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao", "torchaudio"])

print("Done. Now restart the runtime: Runtime > Restart session, then run from cell 2.")
"""


## 1b. Energy & GPU Monitoring Libraries


In [ ]:
# Energy & GPU monitoring libraries.
# codecarbon measures real GPU energy via NVML; pynvml enables per-step power readings.
# Run once (no kernel restart needed).

#%pip install -q codecarbon pynvml

## 2. Runtime Configuration

In [ ]:
import os
from pathlib import Path

def list_dir(path, indent=0):
    p = Path(path)
    pad = "  " * indent
    if not p.exists():
        print(f"{pad}(missing) {path}")
        return
    if not p.is_dir():
        print(f"{pad}{p.name}")
        return
    print(f"{pad}{p}/")
    try:
        for child in sorted(p.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower())):
            if child.is_dir():
                list_dir(child, indent + 1)
            else:
                print(f"{pad}  {child.name}")
    except PermissionError as e:
        print(f"{pad}(permission error: {e})")

# Root: what Kaggle mounted
print("=== /kaggle/input ===")
inp = Path("/kaggle/input")
if inp.exists():
    for top in sorted(inp.iterdir(), key=lambda x: x.name.lower()):
        print(f"  {top.name}/")
else:
    print("  (not found — not on Kaggle or path wrong)")

# Your path — update to match the dataset slug you attached on Kaggle
target = "/kaggle/input/your-clouddataset/CloudDataset_Fixed"
print(f"\n=== exists? {os.path.exists(target)} ===\n{target}")

# If wrong, search for similar folder names
if inp.exists():
    print("\n=== paths containing 'cloud' or 'fixed' (max 40) ===")
    n = 0
    for p in inp.rglob("*"):
        if n >= 40:
            break
        if p.is_dir() and ("cloud" in p.name.lower() or "fixed" in p.name.lower()):
            print(f"  {p}")
            n += 1

In [ ]:
import ctypes
import glob
import os
import sys
import types
from pathlib import Path

os.environ["PYTORCH_JIT"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import site as _site
_search_patterns = [
    "/usr/local/cuda*/lib64/libnvrtc-builtins.so*",
    "/usr/local/cuda*/lib64/libnvJitLink.so*",
    "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*",
    "/usr/lib/x86_64-linux-gnu/libnvJitLink.so*",
]
for _sp in _site.getsitepackages():
    _search_patterns.append(os.path.join(_sp, "nvidia", "*", "lib", "libnvrtc-builtins.so*"))
    _search_patterns.append(os.path.join(_sp, "nvidia", "*", "lib", "libnvJitLink.so*"))
for _pattern in _search_patterns:
    for _lib in sorted(glob.glob(_pattern)):
        try:
            ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
            print(f"Loaded: {_lib}")
        except OSError:
            pass

import torch

if hasattr(torch._C, "_jit_set_nvfuser_enabled"):
    torch._C._jit_set_nvfuser_enabled(False)

try:
    import triton.backends as _tb
    if not hasattr(_tb, "compiler"):
        _tb.compiler = types.SimpleNamespace(AttrsDescriptor=None)
except ImportError:
    pass

RESTART_SENTINEL = "/content/qwen35_grpo_ranking_restart.txt"
if os.path.exists(RESTART_SENTINEL):
    with open(RESTART_SENTINEL, "r", encoding="utf-8") as handle:
        install_pid = handle.read().strip()
    if install_pid == str(os.getpid()):
        raise RuntimeError("Runtime not restarted. Use Runtime > Restart session.")
    os.remove(RESTART_SENTINEL)

import numpy as np
import transformers

# Newer transformers dev builds no longer expose BloomPreTrainedModel, which peft
# imports at module load. Provide a dummy so `import peft` succeeds.
try:
    transformers.BloomPreTrainedModel
except Exception:
    transformers.BloomPreTrainedModel = type("BloomPreTrainedModel", (), {})

import peft

print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    gpu_props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_props.total_memory / (1024**3):.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── Change INITIAL_LORA_EPOCH to the epoch of the adapter you are loading ────
MODEL_NAME          = "Qwen/Qwen3.5-0.8B"
DATASET_PATH        = "/kaggle/input/your-clouddataset/CloudDataset_Fixed"  # <-- SET THIS
INITIAL_LORA_EPOCH  = 0     # epoch of the adapter being loaded (change per run)
ADDITIONAL_LORA_EPOCHS = 5  # how many additional epochs to train
FINAL_LORA_EPOCH    = INITIAL_LORA_EPOCH + ADDITIONAL_LORA_EPOCHS
OUTPUT_DIR          = f"/kaggle/working/qwen35-08-cloud-classifier-lora-epoch{FINAL_LORA_EPOCH}"
SFT_OUTPUT_DIR      = os.path.join(OUTPUT_DIR, f"sft_epoch{INITIAL_LORA_EPOCH}_to_epoch{FINAL_LORA_EPOCH}")
FINAL_ADAPTER_DIR   = os.path.join(SFT_OUTPUT_DIR, f"lora_adapter_epoch{FINAL_LORA_EPOCH}")
MAX_LENGTH          = 768
SEED                = 42
CONTINUATION_LEARNING_RATE = 1e-4

# Path to the adapter from the previous run. Update the slug to match your
# Kaggle input dataset for the previous epoch's adapter.
# If left as the placeholder, auto-detection below will search for it.
PREVIOUS_ADAPTER_PATH = (
    "/kaggle/input/your-previous-lora-adapter"  # <-- SET THIS
)

_inp = Path("/kaggle/input")
if _inp.exists():
    print("\n=== /kaggle/input datasets ===")
    for d in sorted(_inp.iterdir()):
        print(f"  {d.name}/")
    if not os.path.isdir(PREVIOUS_ADAPTER_PATH):
        # Auto-detect: search for any adapter_model.safetensors under /kaggle/input
        candidates = list(_inp.rglob("adapter_model.safetensors"))
        if candidates:
            PREVIOUS_ADAPTER_PATH = str(candidates[0].parent)
            print(f"\nAuto-detected adapter: {PREVIOUS_ADAPTER_PATH}")

    if not os.path.isdir(DATASET_PATH):
        for p in _inp.rglob("CloudDataset_Fixed"):
            if p.is_dir():
                DATASET_PATH = str(p)
                print(f"Auto-detected dataset: {DATASET_PATH}")
                break

torch.manual_seed(SEED)
np.random.seed(SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SFT_OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

print(f"Model:             {MODEL_NAME}")
print(f"Dataset:           {DATASET_PATH}")
print(f"Previous adapter:  epoch {INITIAL_LORA_EPOCH} -> {PREVIOUS_ADAPTER_PATH}")
print(f"Additional epochs: {ADDITIONAL_LORA_EPOCHS}")
print(f"Final adapter:     epoch {FINAL_LORA_EPOCH} -> {FINAL_ADAPTER_DIR}")
print(f"Learning rate:     {CONTINUATION_LEARNING_RATE:.2e}")

assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
assert os.path.isdir(PREVIOUS_ADAPTER_PATH), (
    f"Previous adapter not found: {PREVIOUS_ADAPTER_PATH}\n"
    "Attach the Kaggle dataset containing adapter_model.safetensors and rerun this cell."
)


## 3. Load and Merge Dataset Classes

In [ ]:
from datasets import ClassLabel, load_dataset

print("Loading dataset...")
dataset = load_dataset(
    "imagefolder",
    data_files={
        "train": f"{DATASET_PATH}/train/**",
        "validation": f"{DATASET_PATH}/val/**",
        "test": f"{DATASET_PATH}/test/**",
    },
)

original_labels = dataset["train"].features["label"].names
print(f"Original classes ({len(original_labels)}): {original_labels}")

# WMO-aligned taxonomy: each descriptive label is merged into its closest WMO genus.
class_mapping = {
    "Patterned Clouds":   "Altocumulus",    # wave/ripple pattern = Ac
    "Sky (General)":      "Clear Sky",      # both = no significant cloud cover
    "Thick Dark Clouds":  "Cumulonimbus",   # dark storm cloud = Cb
    "Thick White Clouds": "Cumulus",        # tall white cloud = Cu
    "Veil Clouds":        "Veil",           # thin semi-transparent layer
}

merged_labels = sorted({class_mapping.get(label, label) for label in original_labels})
label2id = {label: i for i, label in enumerate(merged_labels)}
NUM_LABELS = len(merged_labels)

def remap_example(example):
    original_name = original_labels[example["label"]]
    merged_name = class_mapping.get(original_name, original_name)
    example["label"] = label2id[merged_name]
    return example

dataset = dataset.map(remap_example)
new_features = dataset["train"].features.copy()
new_features["label"] = ClassLabel(names=merged_labels)
dataset = dataset.cast(new_features)

new_labels = dataset["train"].features["label"].names
label2id = {label: i for i, label in enumerate(new_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Merged classes ({NUM_LABELS}): {new_labels}")
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")


## 4. Define Classification Prompt & SFT Targets


In [ ]:
SYSTEM_PROMPT = (
    "You are a cloud classification expert with a lot of expertise on weather, physics, and clouds. "
    "Your goal is to, given a cloud image, classify it with one of the following classes:\n\n"
    + "\n".join(f"- {label}" for label in new_labels)
)

USER_PROMPT = (
    "Classify the cloud type in this image. "
    "The class must be included between the tags <answer></answer> using exactly one "
    "of the class names listed above. The tags must be placed at the end of your answer."
)

ANSWER_TEMPLATE = "<answer>{label}</answer>"

def build_messages(label_text=None):
    """Build chat messages. If label_text is given, append the assistant answer target."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": USER_PROMPT},
        ]},
    ]
    if label_text is not None:
        answer = ANSWER_TEMPLATE.format(label=label_text)
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        )
    return messages

# Preview
print("=== Sample prompt ===")
sample_msgs = build_messages("Cumulonimbus")
for msg in sample_msgs:
    texts = [c["text"] for c in msg["content"] if c["type"] == "text"]
    has_img = any(c["type"] == "image" for c in msg["content"])
    print(f"\n[{msg['role']}] {'[IMG] ' if has_img else ''}")
    for t in texts:
        print(t)


## 5. Continue LoRA SFT Classification

Load the base Qwen3.5-0.8B model, attach the **previous LoRA adapter** as trainable,
and continue SFT for 5 more epochs on the classification targets.


In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

compute_dtype = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32

print("Loading base model...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda":
    base_model = base_model.to(DEVICE)

print(f"Loading trainable epoch-{INITIAL_LORA_EPOCH} LoRA adapter from {PREVIOUS_ADAPTER_PATH} ...")
model = PeftModel.from_pretrained(base_model, PREVIOUS_ADAPTER_PATH, is_trainable=True)
model.print_trainable_parameters()


## 5b. Memory Breakdown - LoRA vs Full Fine-Tuning


In [ ]:
import math


def detailed_memory_report(model, label=""):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total     = trainable + frozen
    # All params stored in bf16 (2 bytes each)
    # Adam keeps: fp32 master weight + momentum m + variance v = 3 x 4 bytes per trainable param
    # Gradients: 1 x fp32 = 4 bytes per trainable param

    model_gb         = total     * 2 / 1e9
    grad_lora_gb     = trainable * 4 / 1e9
    grad_full_ft_gb  = total     * 4 / 1e9
    adam_lora_gb     = trainable * 3 * 4 / 1e9
    adam_full_ft_gb  = total     * 3 * 4 / 1e9
    total_lora_gb    = model_gb + grad_lora_gb  + adam_lora_gb
    total_full_ft_gb = model_gb + grad_full_ft_gb + adam_full_ft_gb

    print(f"\n{'='*67}")
    print(f"  Memory Breakdown - {label}")
    print(f"{'='*67}")
    print(f"  {'Component':<44} {'LoRA':>9}  {'Full FT':>9}")
    print(f"  {'-'*44} {'-'*9}  {'-'*9}")
    print(f"  {'Model weights (bf16)':<44} {model_gb:>8.2f}G  {model_gb:>8.2f}G")
    print(f"  {'Gradients (fp32, trainable only)':<44} {grad_lora_gb*1000:>7.0f}MB  {grad_full_ft_gb:>8.2f}G")
    print(f"  {'Adam states (m + v + fp32 master)':<44} {adam_lora_gb*1000:>7.0f}MB  {adam_full_ft_gb:>8.2f}G")
    print(f"  {'-'*44} {'-'*9}  {'-'*9}")
    print(f"  {'TOTAL (theoretical minimum)':<44} {total_lora_gb:>8.2f}G  {total_full_ft_gb:>8.2f}G")
    print(f"\n  Memory saving vs full fine-tuning  : {(1 - total_lora_gb/total_full_ft_gb)*100:.1f}%")
    print(f"  Full FT would need at least        : {math.ceil(total_full_ft_gb)} GB VRAM")
    print(f"  Tesla T4 available                 : 14.6 GB")
    print(f"  LoRA fits on T4?                   : {'YES' if total_lora_gb <= 14.6 else 'NO'}")
    print(f"  Full FT fits on T4?                : {'YES' if total_full_ft_gb <= 14.6 else 'NO - needs a larger GPU'}")

    if torch.cuda.is_available():
        alloc_gb = torch.cuda.memory_allocated() / 1e9
        rsvd_gb  = torch.cuda.memory_reserved()  / 1e9
        print(f"\n  [Actual GPU] Allocated after load  : {alloc_gb:.2f} GB")
        print(f"  [Actual GPU] Reserved after load   : {rsvd_gb:.2f} GB")

    return total_lora_gb, total_full_ft_gb


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
lora_mem_gb, full_ft_mem_gb = detailed_memory_report(model, label="Qwen3.5-0.8B + LoRA (SFT)")


In [ ]:
def apply_template(messages, add_generation_prompt):
    return processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=add_generation_prompt,
    )

class QwenVLMCollator:
    """Data collator that creates single-label classification targets for SFT."""
    def __init__(self, processor, max_length):
        self.processor = processor
        self.max_length = max_length

    def __call__(self, examples):
        images, prompt_texts, full_texts = [], [], []
        for ex in examples:
            image = ex["image"].convert("RGB")
            label_text = id2label[int(ex["label"])]
            prompt_msgs = build_messages()
            full_msgs = build_messages(label_text=label_text)
            prompt_texts.append(apply_template(prompt_msgs, add_generation_prompt=True))
            full_texts.append(apply_template(full_msgs, add_generation_prompt=False))
            images.append(image)

        prompt_batch = self.processor(text=prompt_texts, images=images, padding=True,
                                      truncation=True, max_length=self.max_length, return_tensors="pt")
        full_batch = self.processor(text=full_texts, images=images, padding=True,
                                    truncation=True, max_length=self.max_length, return_tensors="pt")

        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        for i in range(labels.shape[0]):
            prompt_len = int(prompt_batch["attention_mask"][i].sum().item())
            labels[i, :prompt_len] = -100
        full_batch["labels"] = labels
        return full_batch

collator = QwenVLMCollator(processor=processor, max_length=MAX_LENGTH)
sample_batch = collator([dataset["train"][0]])
print("Batch shapes:", {k: tuple(v.shape) for k, v in sample_batch.items()})
print(f"Masked tokens: {(sample_batch['labels'] == -100).sum().item()}")
print(f"Target tokens: {(sample_batch['labels'] != -100).sum().item()}")


## 5c. Efficiency Callback - per-step timing, VRAM & power


In [ ]:
import time
from transformers import TrainerCallback

EFF_LABEL = f"SFT epoch{INITIAL_LORA_EPOCH}to{FINAL_LORA_EPOCH}"

# pynvml: grab handles for ALL available GPUs.
try:
    import pynvml
    pynvml.nvmlInit()
    _N_GPUS_NVML   = pynvml.nvmlDeviceGetCount()
    _nvml_handles  = [pynvml.nvmlDeviceGetHandleByIndex(i) for i in range(_N_GPUS_NVML)]
    _NVML_AVAILABLE = True
    print(f"pynvml available: monitoring {_N_GPUS_NVML} GPU(s) for real-time power.")
except Exception:
    _nvml_handles   = []
    _NVML_AVAILABLE = False
    print("pynvml not available: power will be estimated from TDP after training.")

_N_CUDA_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f"PyTorch sees {_N_CUDA_GPUS} CUDA device(s).")


class EfficiencyCallback(TrainerCallback):
    """
    Records per-step timing, GPU VRAM, and GPU power draw during training.
    All VRAM figures are SUMMED across every visible GPU so multi-GPU runs
    (e.g. T4 x2) are measured correctly rather than reading only GPU 0.
    Power figures are also summed across all GPUs.
    """

    def __init__(self):
        self.step_times    = []
        self.mem_alloc     = []   # total allocated across all GPUs
        self.mem_reserved  = []   # total reserved  across all GPUs
        self.mem_per_gpu   = []   # list of per-GPU allocated lists, one entry per step
        self.power_w       = []   # total power across all GPUs (W)
        self.power_per_gpu = []   # list of per-GPU power lists, one entry per step
        self._step_start   = None
        self._train_start  = None
        self.peak_mem_gb   = 0.0  # peak TOTAL across all GPUs

    def on_train_begin(self, args, state, control, **kwargs):
        self._train_start = time.perf_counter()
        if torch.cuda.is_available():
            for i in range(_N_CUDA_GPUS):
                torch.cuda.reset_peak_memory_stats(i)

    def on_step_begin(self, args, state, control, **kwargs):
        self._step_start = time.perf_counter()

    def on_step_end(self, args, state, control, **kwargs):
        if self._step_start is not None:
            self.step_times.append(time.perf_counter() - self._step_start)

        if torch.cuda.is_available():
            per_gpu_alloc = [torch.cuda.memory_allocated(i) / 1e9 for i in range(_N_CUDA_GPUS)]
            per_gpu_rsvd  = [torch.cuda.memory_reserved(i)  / 1e9 for i in range(_N_CUDA_GPUS)]
            self.mem_alloc.append(sum(per_gpu_alloc))
            self.mem_reserved.append(sum(per_gpu_rsvd))
            self.mem_per_gpu.append(per_gpu_alloc)
            total_peak = sum(torch.cuda.max_memory_allocated(i) for i in range(_N_CUDA_GPUS)) / 1e9
            self.peak_mem_gb = max(self.peak_mem_gb, total_peak)

        if _NVML_AVAILABLE:
            try:
                per_gpu_pwr = [pynvml.nvmlDeviceGetPowerUsage(h) / 1000 for h in _nvml_handles]
                self.power_w.append(sum(per_gpu_pwr))
                self.power_per_gpu.append(per_gpu_pwr)
            except Exception:
                pass

    def on_train_end(self, args, state, control, **kwargs):
        self.total_train_time = time.perf_counter() - self._train_start

    def summary(self):
        import numpy as _np
        n_gpus = max(_N_CUDA_GPUS, 1)
        if torch.cuda.is_available():
            vram_per_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
        else:
            vram_per_gpu = 0.0
        total_vram = vram_per_gpu * n_gpus

        print("\n" + "=" * 65)
        print(f"  Training Efficiency Summary ({n_gpus}x GPU, all GPUs measured)")
        print("=" * 65)

        if self.step_times:
            st = _np.array(self.step_times)
            print(f"  Steps completed              : {len(st)}")
            print(f"  Avg step time                : {st.mean()*1000:.1f} ms")
            print(f"  Median step time             : {_np.median(st)*1000:.1f} ms")
            print(f"  Min / Max step time          : {st.min()*1000:.1f} / {st.max()*1000:.1f} ms")

        if self.mem_alloc:
            ma = _np.array(self.mem_alloc)
            print(f"\n  --- VRAM (total across {n_gpus} GPUs) ---")
            print(f"  Avg total GPU mem allocated  : {ma.mean():.2f} GB")
            print(f"  Peak total GPU mem allocated : {self.peak_mem_gb:.2f} GB")
            if total_vram > 0:
                print(f"  Total VRAM available         : {total_vram:.1f} GB  ({n_gpus}x {vram_per_gpu:.1f} GB)")
                print(f"  Peak / total VRAM            : {self.peak_mem_gb/total_vram*100:.1f}%")
            if self.mem_per_gpu:
                last = self.mem_per_gpu[-1]
                print(f"\n  --- Per-GPU VRAM (last step) ---")
                for gi, v in enumerate(last):
                    print(f"  GPU {gi}                         : {v:.2f} GB")

        if self.power_w:
            pw = _np.array(self.power_w)
            tdp_total = 70 * n_gpus
            print(f"\n  --- Power (total across {n_gpus} GPUs) ---")
            print(f"  Avg total GPU power draw     : {pw.mean():.1f} W")
            print(f"  Max total GPU power draw     : {pw.max():.1f} W")
            print(f"  Combined TDP ({n_gpus}x 70 W)      : {tdp_total} W")
            print(f"  Avg utilisation of TDP       : {pw.mean()/tdp_total*100:.1f}%")
            if self.power_per_gpu:
                last_pwr = self.power_per_gpu[-1]
                print(f"\n  --- Per-GPU Power (last step) ---")
                for gi, p in enumerate(last_pwr):
                    print(f"  GPU {gi}                         : {p:.1f} W  ({p/70*100:.0f}% of TDP)")

        if hasattr(self, "total_train_time"):
            print(f"\n  Total training time          : {self.total_train_time/60:.2f} min")


efficiency_cb = EfficiencyCallback()
print("EfficiencyCallback ready.")


In [ ]:
from transformers import Trainer, TrainingArguments

training_cuda = DEVICE == "cuda"
training_bf16 = training_cuda and torch.cuda.is_bf16_supported()
training_fp16 = training_cuda and not training_bf16

_token_accuracy_totals = {"correct": 0, "total": 0}


def compute_token_accuracy(eval_pred, compute_result=False):
    """Token accuracy on non-masked (assistant) positions; logged as eval_accuracy per epoch."""
    predictions = eval_pred.predictions
    labels = eval_pred.label_ids
    if predictions is None:
        if compute_result:
            _token_accuracy_totals["correct"] = 0
            _token_accuracy_totals["total"] = 0
            return {"accuracy": 0.0}
        return {}
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    if hasattr(predictions, "detach"):
        predictions = predictions.detach().cpu().float().numpy()
    else:
        predictions = np.asarray(predictions)
    if hasattr(labels, "detach"):
        labels = labels.detach().cpu().numpy()
    else:
        labels = np.asarray(labels)
    if predictions.ndim != 3:
        if compute_result:
            c, t = _token_accuracy_totals["correct"], _token_accuracy_totals["total"]
            _token_accuracy_totals["correct"] = 0
            _token_accuracy_totals["total"] = 0
            return {"accuracy": float(c / t) if t > 0 else 0.0}
        return {}
    if predictions.shape[1] < 2 or labels.shape[1] < 2:
        if compute_result:
            c, t = _token_accuracy_totals["correct"], _token_accuracy_totals["total"]
            _token_accuracy_totals["correct"] = 0
            _token_accuracy_totals["total"] = 0
            return {"accuracy": float(c / t) if t > 0 else 0.0}
        return {}
    shift_logits = predictions[:, :-1, :]
    shift_labels = labels[:, 1:]
    preds = np.argmax(shift_logits, axis=-1)
    mask = shift_labels != -100
    _token_accuracy_totals["correct"] += int(np.sum((preds == shift_labels) & mask))
    _token_accuracy_totals["total"] += int(np.sum(mask))
    if compute_result:
        c, t = _token_accuracy_totals["correct"], _token_accuracy_totals["total"]
        _token_accuracy_totals["correct"] = 0
        _token_accuracy_totals["total"] = 0
        return {"accuracy": float(c / t) if t > 0 else 0.0}
    return {}


sft_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=1,
    num_train_epochs=ADDITIONAL_LORA_EPOCHS,
    learning_rate=CONTINUATION_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=training_bf16,
    fp16=training_fp16,
    gradient_checkpointing=False,
    remove_unused_columns=False,
    dataloader_num_workers=2 if training_cuda else 0,
    dataloader_pin_memory=training_cuda,
    prediction_loss_only=False,
    batch_eval_metrics=True,
    report_to="none",
    seed=SEED,
)

sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=collator,
    compute_metrics=compute_token_accuracy,
    callbacks=[efficiency_cb],
)

# CodeCarbon energy tracker around training.
try:
    from codecarbon import EmissionsTracker
    _tracker = EmissionsTracker(
        project_name=f"qwen35_sft_epoch{FINAL_LORA_EPOCH}",
        output_dir=SFT_OUTPUT_DIR,
        log_level="error",
        save_to_file=True,
    )
    _tracker.start()
    _use_codecarbon = True
except Exception as _cc_err:
    print(f"CodeCarbon unavailable ({_cc_err}), will use TDP estimate.")
    _use_codecarbon = False

import time as _time_mod
_t0 = _time_mod.perf_counter()

print(f"Continuing LoRA SFT from epoch {INITIAL_LORA_EPOCH} to epoch {FINAL_LORA_EPOCH}...")
sft_result = sft_trainer.train()
sft_trainer.save_state()
print(f"LoRA continuation done. Loss: {sft_result.training_loss:.4f}")

# Persist the continued LoRA adapter weights.
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)
model.save_pretrained(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Continued LoRA adapter saved to {FINAL_ADAPTER_DIR}")

# Post-training resource metrics.
_elapsed_s   = _time_mod.perf_counter() - _t0
_elapsed_min = _elapsed_s / 60
_n_gpus      = torch.cuda.device_count() if torch.cuda.is_available() else 0

if _n_gpus > 0:
    _vram_per_gpu     = torch.cuda.get_device_properties(0).total_memory / 1e9
    _total_vram       = _vram_per_gpu * _n_gpus
    _peak_mem_gb      = sum(torch.cuda.max_memory_allocated(i) for i in range(_n_gpus)) / 1e9
    _peak_mem_per_gpu = [torch.cuda.max_memory_allocated(i) / 1e9 for i in range(_n_gpus)]
else:
    _vram_per_gpu = _total_vram = _peak_mem_gb = 0.0
    _peak_mem_per_gpu = []

if _use_codecarbon:
    try:
        _emissions_kg = _tracker.stop()
        _energy_kwh   = _tracker._total_energy.kWh
    except Exception:
        _use_codecarbon = False

if not _use_codecarbon:
    _T4_TDP_W     = 70
    _util         = 0.85
    _energy_kwh   = (_T4_TDP_W * _util * max(_n_gpus, 1) * _elapsed_s / 3600) / 1000
    _emissions_kg = _energy_kwh * 0.233  # EU average grid: 233 g CO2/kWh

print("\n" + "=" * 65)
print("  Post-Training Resource Metrics")
print("=" * 65)
print(f"  Total training time          : {_elapsed_min:.2f} min  ({_elapsed_s:.0f} s)")
print(f"  Number of GPUs used          : {_n_gpus}")
if _n_gpus > 0:
    print(f"  Peak VRAM - total            : {_peak_mem_gb:.2f} GB  /  {_total_vram:.1f} GB  ({_peak_mem_gb/_total_vram*100:.1f}%)")
    for _gi, _v in enumerate(_peak_mem_per_gpu):
        print(f"  Peak VRAM - GPU {_gi}            : {_v:.2f} GB  /  {_vram_per_gpu:.1f} GB  ({_v/_vram_per_gpu*100:.1f}%)")
print(f"  Energy consumed              : {_energy_kwh:.4f} kWh")
print(f"  CO2 equivalent               : {_emissions_kg*1000:.2f} g CO2eq")
_method = "CodeCarbon (NVML, all GPUs)" if _use_codecarbon else f"Manual TDP estimate (70 W x 85% x {max(_n_gpus,1)} GPUs)"
print(f"  Energy measurement           : {_method}")
_metrics = sft_result.metrics
print(f"\n  Samples/sec                  : {_metrics.get('train_samples_per_second', 'N/A')}")
print(f"  Steps/sec                    : {_metrics.get('train_steps_per_second', 'N/A')}")
print(f"  Trainer runtime (s)          : {_metrics.get('train_runtime', 'N/A')}")

efficiency_cb.summary()


In [ ]:
# Zip the continued LoRA adapter and full output into /kaggle/working/.
import os
import shutil

_zip_out_dir = "/kaggle/working"
os.makedirs(_zip_out_dir, exist_ok=True)

if os.path.isdir(FINAL_ADAPTER_DIR):
    adapter_zip = os.path.join(_zip_out_dir, f"qwen35_lora_epoch{FINAL_LORA_EPOCH}_adapter")
    shutil.make_archive(adapter_zip, "zip", root_dir=FINAL_ADAPTER_DIR)
    print(f"Adapter zip: {adapter_zip}.zip")
else:
    print(f"Skipped adapter zip; missing: {FINAL_ADAPTER_DIR}")

if os.path.isdir(OUTPUT_DIR):
    full_zip = os.path.join(_zip_out_dir, f"qwen35_lora_epoch{FINAL_LORA_EPOCH}_full")
    shutil.make_archive(full_zip, "zip", root_dir=OUTPUT_DIR)
    print(f"Full output zip: {full_zip}.zip")


## 7. Evaluate on Test Set

In [ ]:
import re
from tqdm.auto import tqdm

def move_to(batch, device):
    return {k: v.to(device) if hasattr(v, "to") else v for k, v in batch.items()}

GENERATION_KWARGS = {"max_new_tokens": 64, "do_sample": False}

def classify_image(image):
    """Run inference and return the raw text output."""
    image = image.convert("RGB")
    prompt_text = apply_template(build_messages(), add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], padding=True, return_tensors="pt")
    inputs = move_to(inputs, next(model.parameters()).device)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, **GENERATION_KWARGS)
    gen_ids = gen_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

_exact_match = {label.lower(): label for label in new_labels}

def normalize_prediction(text):
    """Extract the predicted class from <answer></answer> tags, with light fallbacks."""
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    if m:
        text = m.group(1).strip()
    cleaned = " ".join(text.replace("\n", " ").split()).strip().lower()
    if cleaned in _exact_match:
        return _exact_match[cleaned]
    for label in new_labels:
        if label.lower() in cleaned or cleaned in label.lower():
            return label
    cleaned_tokens = set(cleaned.replace("/", " ").replace("_", " ").replace("-", " ").split())
    best_label, best_score = None, 0
    for label in new_labels:
        label_tokens = set(label.lower().replace("_", " ").replace("-", " ").split())
        score = len(cleaned_tokens & label_tokens)
        if score > best_score:
            best_score, best_label = score, label
    return best_label if best_label is not None else new_labels[0]

def has_valid_answer(text):
    """True if the output contains an <answer> tag holding a valid class name."""
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    if not m:
        return False
    return " ".join(m.group(1).split()).strip().lower() in _exact_match

model.eval()
true_labels_list, pred_labels_list, raw_responses = [], [], []

test_split = dataset["test"]
for example in tqdm(test_split, total=len(test_split)):
    raw = classify_image(example["image"])
    pred = normalize_prediction(raw)
    true_label = id2label[int(example["label"])]
    true_labels_list.append(true_label)
    pred_labels_list.append(pred)
    raw_responses.append(raw)

accuracy_top1 = sum(t == p for t, p in zip(true_labels_list, pred_labels_list)) / len(true_labels_list)
format_rate = sum(has_valid_answer(r) for r in raw_responses) / len(raw_responses)

print(f"\n{'='*50}")
print(f"Top-1 Accuracy: {accuracy_top1:.4f}")
print(f"Format rate:    {format_rate:.4f}")
print(f"{'='*50}")

print("\n--- Sample Predictions ---")
for i in range(min(5, len(raw_responses))):
    correct = '\u2713' if true_labels_list[i] == pred_labels_list[i] else '\u2717'
    print(f"{correct} True: {true_labels_list[i]:15s} | Pred: {pred_labels_list[i]:15s}")


## 8. Classification Report & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

report = classification_report(true_labels_list, pred_labels_list, labels=new_labels, digits=3, zero_division=0)
print(report)

cm = confusion_matrix(true_labels_list, pred_labels_list, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix - Qwen3.5-0.8B LoRA Epoch {FINAL_LORA_EPOCH} Classification (Top-1 Acc: {accuracy_top1:.3f})")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_lora_epoch{FINAL_LORA_EPOCH}_classification.png")
plt.savefig(cm_path, dpi=150)
plt.show()

predictions_df = pd.DataFrame({
    "true_label": true_labels_list,
    "predicted_label": pred_labels_list,
    "raw_response": raw_responses,
})
pred_path = os.path.join(OUTPUT_DIR, f"test_predictions_lora_epoch{FINAL_LORA_EPOCH}_classification.csv")
predictions_df.to_csv(pred_path, index=False)
print(f"Saved predictions to {pred_path}")

per_class = (
    predictions_df.assign(correct=predictions_df["true_label"] == predictions_df["predicted_label"])
    .groupby("true_label")["correct"].mean()
    .reindex(new_labels)
)
fig, ax = plt.subplots(figsize=(12, 6))
per_class.plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Accuracy")
ax.set_title(f"Per-Class Accuracy - LoRA Epoch {FINAL_LORA_EPOCH}")
ax.axvline(1 / NUM_LABELS, color="red", ls="--", label="Random baseline")
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
pc_path = os.path.join(OUTPUT_DIR, f"per_class_accuracy_lora_epoch{FINAL_LORA_EPOCH}.png")
plt.savefig(pc_path, dpi=150)
plt.show()


## 9. Save Artifacts

In [ ]:
metrics_path = os.path.join(OUTPUT_DIR, f"metrics_summary_epoch{FINAL_LORA_EPOCH}.txt")
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write(f"accuracy_top1={accuracy_top1:.6f}\n")
    f.write(f"format_rate={format_rate:.6f}\n")
    f.write(f"num_test_samples={len(true_labels_list)}\n")
    f.write(f"model={MODEL_NAME}\n")
    f.write(f"method=LoRA_SFT_classification_epoch{INITIAL_LORA_EPOCH}_to_epoch{FINAL_LORA_EPOCH}\n")

misclassified_df = predictions_df[predictions_df["true_label"] != predictions_df["predicted_label"]]
print(f"Saved metrics to {metrics_path}")
print(f"Final adapter: {FINAL_ADAPTER_DIR}")
print(f"Misclassified (top-1): {len(misclassified_df)}")
print(f"\n=== Final Results ===")
print(f"Top-1 Accuracy: {accuracy_top1:.4f}")
print(f"Format rate:    {format_rate:.4f}")


## 9b. LoRA Storage Comparison


In [ ]:
from pathlib import Path


def dir_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6

adapter_size_mb = dir_size_mb(FINAL_ADAPTER_DIR) if os.path.isdir(FINAL_ADAPTER_DIR) else 0.0
_total_params = sum(p.numel() for p in model.parameters())
full_ckpt_mb  = _total_params * 2 / 1e6  # bf16

print("\n" + "=" * 60)
print("  Storage / Checkpoint Comparison")
print("=" * 60)
print(f"  LoRA adapter on disk    : {adapter_size_mb:.1f} MB")
print(f"  Full model checkpoint   : ~{full_ckpt_mb:.0f} MB  (theoretical, bf16)")
if adapter_size_mb > 0:
    print(f"  Storage reduction       : {full_ckpt_mb / adapter_size_mb:.0f}x smaller")
    print(f"  Percentage of full ckpt : {adapter_size_mb / full_ckpt_mb * 100:.2f}%")


## 9c. Efficiency Plots: Step Time, VRAM, Power


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Step time
ax = axes[0]
if efficiency_cb.step_times:
    steps_arr = np.arange(1, len(efficiency_cb.step_times) + 1)
    ax.plot(steps_arr, np.array(efficiency_cb.step_times) * 1000, alpha=0.7, linewidth=0.8)
    mean_ms = np.mean(efficiency_cb.step_times) * 1000
    ax.axhline(mean_ms, color="red", linestyle="--", label=f"Mean: {mean_ms:.0f} ms")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("Step Time (ms)")
    ax.set_title("Step Time per Step")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No step-time data\n(EfficiencyCallback not active)", ha="center", va="center", transform=ax.transAxes)

# GPU VRAM
ax = axes[1]
if efficiency_cb.mem_alloc:
    steps_mem = np.arange(1, len(efficiency_cb.mem_alloc) + 1)
    ax.plot(steps_mem, efficiency_cb.mem_alloc,    label="Allocated", alpha=0.8)
    ax.plot(steps_mem, efficiency_cb.mem_reserved, label="Reserved",  alpha=0.8)
    ax.axhline(14.6, color="red", linestyle="--", label="T4 VRAM limit (14.6 GB)")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("GPU Memory (GB)")
    ax.set_title("GPU VRAM Usage During Training")
    ax.legend()
else:
    ax.text(0.5, 0.5, "No VRAM data", ha="center", va="center", transform=ax.transAxes)

# GPU Power
ax = axes[2]
if efficiency_cb.power_w:
    steps_pwr = np.arange(1, len(efficiency_cb.power_w) + 1)
    ax.plot(steps_pwr, efficiency_cb.power_w, alpha=0.8, color="orange")
    ax.axhline(70, color="red", linestyle="--", label="T4 TDP (70 W)")
    ax.axhline(np.mean(efficiency_cb.power_w), color="blue", linestyle="--",
               label=f"Mean: {np.mean(efficiency_cb.power_w):.0f} W")
    ax.set_xlabel("Training Step")
    ax.set_ylabel("GPU Power (W)")
    ax.set_title("GPU Power Draw During Training")
    ax.legend()
else:
    ax.text(0.5, 0.5, "pynvml not available\n(no power data)", ha="center", va="center", transform=ax.transAxes)

plt.suptitle(f"LoRA SFT Training Efficiency - Qwen3.5-0.8B {EFF_LABEL}", fontsize=13, fontweight="bold")
plt.tight_layout()
plots_path = os.path.join(OUTPUT_DIR, f"lora_efficiency_plots_sft_epoch{FINAL_LORA_EPOCH}.png")
plt.savefig(plots_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {plots_path}")


## 9d. Master LoRA Efficiency Summary Table


In [ ]:
import pandas as pd

_trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
_total        = sum(p.numel() for p in model.parameters())
_frozen       = _total - _trainable
_bytes_per_p  = 2  # bf16

_full_weights_gb = _total * _bytes_per_p / 1e9
_full_grad_gb    = _total * 4 / 1e9
_full_adam_gb    = _total * 3 * 4 / 1e9
_full_total_gb   = _full_weights_gb + _full_grad_gb + _full_adam_gb

_lora_grad_gb    = _trainable * 4 / 1e9
_lora_adam_gb    = _trainable * 3 * 4 / 1e9
_lora_total_gb   = _full_weights_gb + _lora_grad_gb + _lora_adam_gb

_peak_val = efficiency_cb.peak_mem_gb if efficiency_cb.peak_mem_gb > 0 else _peak_mem_gb
_time_val = efficiency_cb.total_train_time / 60 if hasattr(efficiency_cb, "total_train_time") else _elapsed_min
_n_gpus   = max(torch.cuda.device_count() if torch.cuda.is_available() else 0, 1)

rows = [
    ("Trainable parameters",          f"{_trainable:,}",                                          f"{_total:,}"),
    ("Frozen parameters",             f"{_frozen:,}",                                             "0"),
    ("% of parameters trained",       f"{_trainable/_total*100:.4f}%",                            "100%"),
    ("Parameter reduction factor",    f"{_total/_trainable:.0f}x fewer trainable params",         "1x (baseline)"),
    ("Model weights (bf16)",          f"~{_full_weights_gb:.2f} GB",                              f"~{_full_weights_gb:.2f} GB"),
    ("Gradient memory",               f"~{_lora_grad_gb*1000:.0f} MB",                            f"~{_full_grad_gb:.2f} GB"),
    ("Adam optimizer states",         f"~{_lora_adam_gb*1000:.0f} MB",                            f"~{_full_adam_gb:.2f} GB"),
    ("Total theoretical VRAM",        f"~{_lora_total_gb:.2f} GB",                                f"~{_full_total_gb:.2f} GB"),
    ("Fits on Tesla T4 (14.6 GB)?",   "YES (per GPU)",                                            "NO - needs >=40 GB GPU"),
    (f"Peak VRAM - total ({_n_gpus}x GPU)", f"{_peak_val:.2f} GB",                               "N/A (not run)"),
    ("Peak VRAM - per GPU",           f"{_peak_val/_n_gpus:.2f} GB  ({_peak_val/_n_gpus/14.6*100:.0f}% of T4)", "N/A (not run)"),
    ("Training time",                 f"{_time_val:.1f} min",                                     "N/A (not run)"),
    ("Energy consumed",               f"{_energy_kwh:.4f} kWh",                                   "N/A (not run)"),
    ("CO2 equivalent",                f"{_emissions_kg*1000:.2f} g CO2eq",                        "N/A (not run)"),
    ("Adapter / checkpoint size",     f"{adapter_size_mb:.1f} MB",                                f"~{full_ckpt_mb:.0f} MB"),
    ("Storage reduction",             f"{full_ckpt_mb/adapter_size_mb:.0f}x smaller" if adapter_size_mb > 0 else "N/A", "baseline"),
]

df = pd.DataFrame(rows, columns=["Metric", "LoRA SFT (this work)", "Full Fine-Tuning (theoretical)"])
print("\n" + "=" * 80)
print(f"  LoRA Efficiency Summary - {EFF_LABEL}")
print("=" * 80)
print(df.to_string(index=False))

summary_path = os.path.join(OUTPUT_DIR, f"lora_efficiency_summary_sft_epoch{FINAL_LORA_EPOCH}.csv")
df.to_csv(summary_path, index=False)
print(f"\nSaved to {summary_path}")


In [ ]:
from pathlib import Path

path = Path('/kaggle/working/')
for p in path.iterdir():
    print(p.name)